# 23CSE301 ML Capstone - Clustering Track (Review 2)
**Dataset:** Breast Cancer Wisconsin (Diagnostic), UCI ML Repository - 569 samples, 30 numeric features.
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

**Algorithms (2):** K-Means and Agglomerative Hierarchical Clustering.

Clustering is unsupervised: the label column is **never used for fitting**. It is used only afterwards (Adjusted Rand Index) to sanity-check the clusters.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
from scipy.cluster.hierarchy import linkage, dendrogram

RS = 42
np.random.seed(RS)
sns.set_theme(style="whitegrid", palette="colorblind")
os.makedirs("data", exist_ok=True); os.makedirs("outputs", exist_ok=True)

## 1. Load and audit the data

In [2]:
bc = load_breast_cancer(as_frame=True)
df = bc.frame.copy()                       # 'target': 0 = malignant, 1 = benign
df.to_csv("data/breast_cancer_wisconsin.csv", index=False)

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()), "| Duplicates:", int(df.duplicated().sum()))
print(df["target"].map({0: "malignant", 1: "benign"}).value_counts())

X = df.drop(columns="target")              # features only
y = df["target"]                           # kept aside, used ONLY for post-hoc validation

Shape: (569, 31)
Missing values: 0 | Duplicates: 0
target
benign       357
malignant    212
Name: count, dtype: int64


## 2. EDA
Observation: many features are strongly correlated (radius, perimeter, area), so the 30 dimensions carry much less than 30 dimensions of information. This is why PCA to 2D will keep a large share of the variance.

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(X.iloc[:, :10].corr(), cmap="coolwarm", ax=ax[0], cbar_kws={"label": "Pearson r"})
ax[0].set_title("Correlation heatmap (mean features)")
sns.boxplot(data=X[["mean radius", "mean texture", "mean smoothness"]], ax=ax[1])
ax[1].set_title("Raw feature scales differ - scaling is required"); ax[1].set_ylabel("Raw value")
plt.tight_layout(); plt.show()

## 3. Scale the features
Both algorithms use Euclidean distance, so an unscaled feature like `area` (hundreds) would dominate `smoothness` (about 0.1). There is no train/test split in clustering, so the scaler is fitted on all feature rows (labels are not involved).

In [4]:
Xs = StandardScaler().fit_transform(X)
print("Mean ~0:", np.round(Xs.mean(), 3), "| Std ~1:", np.round(Xs.std(), 3))

Mean ~0: -0.0 | Std ~1: 1.0


# ALGORITHM 1: K-Means
### Elbow curve (inertia vs k) and silhouette vs k

In [5]:
ks = range(2, 11)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RS).fit(Xs)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xs, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(ks, inertias, "o-"); ax[0].set_title("K-Means elbow curve")
ax[0].set_xlabel("Number of clusters k"); ax[0].set_ylabel("Inertia (within-cluster sum of squares)")
ax[1].plot(ks, sils, "o-", color="C1"); ax[1].set_title("Silhouette score vs k")
ax[1].set_xlabel("Number of clusters k"); ax[1].set_ylabel("Silhouette score")
plt.tight_layout(); plt.show()

The elbow is gradual (no sharp bend), so inertia alone is not decisive. The silhouette score is highest at **k = 2**, so k = 2 is used. It also agrees with the two clinical classes, but that agreement is a check, not the reason.

In [6]:
K = 2
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RS)
km_labels = kmeans.fit_predict(Xs)
print("K-Means cluster sizes:", np.bincount(km_labels), "| iterations:", kmeans.n_iter_)

K-Means cluster sizes: [375 194] | iterations: 7


# ALGORITHM 2: Agglomerative Hierarchical Clustering
### Dendrogram (Ward linkage)

In [7]:
Z = linkage(Xs, method="ward")
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, show_contracted=True)
plt.axhline((Z[-2, 2] + Z[-1, 2]) / 2, color="r", ls="--", label="cut giving k=2")
plt.title("Agglomerative clustering dendrogram (Ward linkage, last 30 merges)")
plt.xlabel("Cluster (size in brackets where merged)"); plt.ylabel("Merge distance")
plt.legend(); plt.tight_layout(); plt.show()

The last two merges (tallest vertical lines) are far apart, so cutting between them gives 2 well-separated groups.

### Compare linkage strategies
Linkage defines how the distance between two clusters is measured: **ward** (minimise increase in within-cluster variance), **complete** (max pairwise distance), **average** (mean pairwise distance), **single** (min pairwise distance).

In [8]:
def metrics(name, labels):
    return {"Algorithm": name,
            "Silhouette": silhouette_score(Xs, labels),
            "Davies-Bouldin": davies_bouldin_score(Xs, labels),
            "Calinski-Harabasz": calinski_harabasz_score(Xs, labels),
            "ARI vs true labels (post-hoc)": adjusted_rand_score(y, labels)}

link_labels = {l: AgglomerativeClustering(n_clusters=K, linkage=l).fit_predict(Xs)
               for l in ["ward", "complete", "average", "single"]}
link_table = pd.DataFrame([metrics(f"Agglomerative ({l})", lab) for l, lab in link_labels.items()])
link_table["Cluster sizes"] = [str(np.bincount(lab)) for lab in link_labels.values()]
link_table.set_index("Algorithm").round(4)

,Silhouette,Davies-Bouldin,Calinski-Harabasz,ARI vs true labels (post-hoc),Cluster sizes
Algorithm,,,,,
Agglomerative (ward),0.3394,1.3700,248.6288,0.5750,[184 385]
Agglomerative (complete),0.6607,0.4497,27.8727,0.0048,[567 2]
Agglomerative (average),0.6340,0.6799,32.4822,0.0073,[566 3]
Agglomerative (single),0.6607,0.4497,27.8727,0.0048,[567 2]


**Read this carefully.** Complete, average and single linkage show a *high* silhouette and *low* Davies-Bouldin, but the cluster sizes show why: they split off 2-3 outlier points and put everything else in one cluster. ARI near 0 confirms they do not capture the real structure. **Ward** is the only linkage that gives two balanced, meaningful clusters, so it is the Agglomerative model used in the final comparison.

## Final comparison: the 2 algorithms
Silhouette and Calinski-Harabasz: higher is better. Davies-Bouldin: lower is better.

In [9]:
agg_labels = link_labels["ward"]
final = pd.DataFrame([metrics("K-Means (k=2)", km_labels),
                      metrics("Agglomerative (Ward, k=2)", agg_labels)]).set_index("Algorithm")
final["Cluster sizes"] = [str(np.bincount(km_labels)), str(np.bincount(agg_labels))]
final.round(4)

,Silhouette,Davies-Bouldin,Calinski-Harabasz,ARI vs true labels (post-hoc),Cluster sizes
Algorithm,,,,,
K-Means (k=2),0.3434,1.3205,267.6917,0.6536,[375 194]
"Agglomerative (Ward, k=2)",0.3394,1.3700,248.6288,0.5750,[184 385]


## Cluster visualisation: PCA (2 components) for both algorithms

In [10]:
pca = PCA(n_components=2, random_state=RS).fit(Xs)
Xp = pca.transform(Xs)
print("Variance explained by PC1+PC2: %.1f%%" % (100 * pca.explained_variance_ratio_.sum()))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, lab) in zip(axes, {"K-Means (k=2)": km_labels, "Agglomerative (Ward, k=2)": agg_labels}.items()):
    sc = ax.scatter(Xp[:, 0], Xp[:, 1], c=lab, cmap="tab10", s=20, alpha=0.8)
    ax.set_title(name)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)")
    ax.legend(*sc.legend_elements(), title="Cluster", loc="best")
plt.tight_layout(); plt.show()

Variance explained by PC1+PC2: 63.2%


## t-SNE visualisation (both algorithms, same embedding)
t-SNE axes have no physical meaning; only which points sit together matters.

In [11]:
Xt = TSNE(n_components=2, perplexity=30, init="pca", random_state=RS).fit_transform(Xs)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, lab) in zip(axes, {"K-Means (k=2)": km_labels, "Agglomerative (Ward, k=2)": agg_labels}.items()):
    sc = ax.scatter(Xt[:, 0], Xt[:, 1], c=lab, cmap="tab10", s=20, alpha=0.8)
    ax.set_title(f"t-SNE - {name}"); ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
    ax.legend(*sc.legend_elements(), title="Cluster", loc="best")
plt.tight_layout(); plt.show()

## Save results (for the README)

In [12]:
final.round(4).to_csv("outputs/clustering_final_comparison.csv")
link_table.round(4).to_csv("outputs/agglomerative_linkage_comparison.csv", index=False)
print("Saved to outputs/")

Saved to outputs/
